In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

### CMIP6 data from Copernicus CDS

**CMIP6** data are accessible through the **Copernicus Climate Data Store (CDS)**.

- **Provider:** Copernicus Climate Change Service (C3S), operated by ECMWF
- **Scenarios:** Historical and future projections (SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5)
- **Spatial resolution:** ~1° or ~0.5° depending on dataset
- **Temporal resolution:** Monthly or daily

> This notebook uses **synthetic demo data** that reproduces the CMIP6/ERA5 format
> without requiring CDS credentials.
> Replace the synthetic section with actual `download_CDS_CMIP6()` calls when credentials are available.

In [2]:
# download_CDS_CMIP6 requires cdsapi + credentials
# Interactive map requires ipyleaflet — both skipped
from pyhydra.data_sources.climate_change import download_CDS_CMIP6
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr
import tempfile, os
print('Imports OK')

Imports OK


---
## download_CDS_CMIP6 — real usage

```python
download_CDS_CMIP6(
    url='https://cds.climate.copernicus.eu/api',
    api_key='<your-key>',
    start_date='1950-01-01',
    end_date='2014-12-31',
    temporal_resolution='monthly',
    model='All',
    experiments=['historical', 'ssp2_4_5', 'ssp5_8_5'],
    variables=['precipitation'],
    download_base_dir='cmip6_data/',
    area=[44, -10, 35, 5],   # [N, W, S, E]
    combinations_csv='combinaciones_validas_monthly.csv',
)
```

In [3]:
# === SYNTHETIC DEMO DATA ===
# Mimics CMIP6 monthly NetCDF: variables pr (kg m-2 s-1), tasmax, tasmin

TMPDIR = tempfile.mkdtemp()
rng = np.random.default_rng(42)

# Historical 1950-2014 + SSP245 2015-2100 — use monthly time steps
times_hist = pd.date_range('1990-01-01', '2014-12-01', freq='MS')
times_ssp  = pd.date_range('2015-01-01', '2060-12-01', freq='MS')
lats  = np.arange(44.0, 34.5, -1.0)   # 1° grid
lons  = np.arange(-10.0, 5.0, 1.0)

def make_cmip6(times, scenario='historical', warming_trend=0.0):
    months_arr = times.month.values[:, None, None]
    year_frac  = ((times.year.values - 1990) * 12 + times.month.values - 1)[:, None, None] / 12
    # pr in kg m-2 s-1 (≈ mm/s), typical ~2e-5
    seasonal_pr = 2e-5 * (1 + 0.6 * np.sin(2 * np.pi * (months_arr - 10) / 12))
    pr = (seasonal_pr * rng.uniform(0.5, 1.5, (len(times), len(lats), len(lons)))).astype('float32')
    # tasmax in K
    seasonal_t = 293 + 12 * np.sin(2 * np.pi * (months_arr - 1) / 12) + warming_trend * year_frac
    tasmax = (seasonal_t + rng.normal(0, 1, (len(times), len(lats), len(lons)))).astype('float32')
    tasmin = tasmax - rng.uniform(5, 15, (len(times), len(lats), len(lons))).astype('float32')
    return xr.Dataset(
        {
            'pr':     (['time', 'lat', 'lon'], pr),
            'tasmax': (['time', 'lat', 'lon'], tasmax),
            'tasmin': (['time', 'lat', 'lon'], tasmin),
        },
        coords={'time': times, 'lat': lats, 'lon': lons},
        attrs={'source': f'Synthetic CMIP6 demo — {scenario}', 'institution': 'DEMO'}
    )

ds_hist = make_cmip6(times_hist, 'historical', warming_trend=0.0)
ds_ssp  = make_cmip6(times_ssp,  'ssp245',     warming_trend=0.04)  # +0.04 K/year

for var in ('pr', 'tasmax', 'tasmin'):
    ds_hist[var].attrs.update({'units': 'kg m-2 s-1' if var == 'pr' else 'K'})
    ds_ssp[var].attrs.update({'units': 'kg m-2 s-1' if var == 'pr' else 'K'})

NC_HIST = os.path.join(TMPDIR, 'cmip6_historical.nc')
NC_SSP  = os.path.join(TMPDIR, 'cmip6_ssp245.nc')
ds_hist.to_netcdf(NC_HIST)
ds_ssp.to_netcdf(NC_SSP)
print(f'Historical NetCDF: {NC_HIST}')
print(f'SSP245 NetCDF:     {NC_SSP}')
print(ds_hist)

Historical NetCDF: /var/folders/44/dw7p6q9108xcc4mmh_f7q1vc0000gn/T/tmpf32miv_k/cmip6_historical.nc
SSP245 NetCDF:     /var/folders/44/dw7p6q9108xcc4mmh_f7q1vc0000gn/T/tmpf32miv_k/cmip6_ssp245.nc
<xarray.Dataset> Size: 543kB
Dimensions:  (time: 300, lat: 10, lon: 15)
Coordinates:
  * time     (time) datetime64[ns] 2kB 1990-01-01 1990-02-01 ... 2014-12-01
  * lat      (lat) float64 80B 44.0 43.0 42.0 41.0 40.0 39.0 38.0 37.0 36.0 35.0
  * lon      (lon) float64 120B -10.0 -9.0 -8.0 -7.0 -6.0 ... 1.0 2.0 3.0 4.0
Data variables:
    pr       (time, lat, lon) float32 180kB 4.077e-05 3.004e-05 ... 1.699e-05
    tasmax   (time, lat, lon) float32 180kB 292.4 292.9 293.5 ... 286.4 289.5
    tasmin   (time, lat, lon) float32 180kB 278.4 278.3 287.6 ... 277.6 281.4
Attributes:
    source:       Synthetic CMIP6 demo — historical
    institution:  DEMO


---
## 1. Extract point time series — Madrid

In [4]:
LAT, LON = 40.0, -4.0   # Madrid

# Combine historical + SSP245
def extract_point(nc_file, lat, lon):
    ds = xr.open_dataset(nc_file)
    pt = ds.sel(lat=lat, lon=lon, method='nearest')
    df = pt.to_dataframe().reset_index()[['time', 'pr', 'tasmax', 'tasmin']]
    df['pr_mm_day'] = df['pr'] * 86400   # kg m-2 s-1 → mm/day
    df['tasmax_c']  = df['tasmax'] - 273.15
    df['tasmin_c']  = df['tasmin'] - 273.15
    ds.close()
    return df.set_index('time').sort_index()

df_hist = extract_point(NC_HIST, LAT, LON)
df_ssp  = extract_point(NC_SSP,  LAT, LON)

print('Historical (first 5 rows):')
print(df_hist[['pr_mm_day', 'tasmax_c', 'tasmin_c']].head())
print('\nSSP245 (first 5 rows):')
print(df_ssp[['pr_mm_day', 'tasmax_c', 'tasmin_c']].head())

Historical (first 5 rows):
            pr_mm_day   tasmax_c   tasmin_c
time                                       
1990-01-01   2.928496  18.291718   4.758179
1990-02-01   1.657091  26.355194  13.353760
1990-03-01   2.862636  30.225800  18.994629
1990-04-01   1.406408  31.743591  23.188019
1990-05-01   1.581469  30.085449  23.411987

SSP245 (first 5 rows):
            pr_mm_day   tasmax_c   tasmin_c
time                                       
2015-01-01   1.631496  22.532715  15.926117
2015-02-01   2.493511  25.836639  16.919495
2015-03-01   2.840103  32.117676  26.264191
2015-04-01   2.187438  32.273895  27.227844
2015-05-01   1.105599  30.777527  21.500214


---
## 2. Visualisation — climate change signal

In [5]:
df_all = pd.concat([df_hist, df_ssp])
annual_tmax = df_all['tasmax_c'].resample('YE').mean()
annual_pr   = df_all['pr_mm_day'].resample('YE').sum()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(annual_tmax.index.year, annual_tmax.values, lw=1.0, color='tomato')
axes[0].axvline(2015, color='gray', ls='--', lw=0.8, label='Historical / SSP245 boundary')
axes[0].set_ylabel('Annual mean Tmax (°C)')
axes[0].set_title('CMIP6 — Madrid — Climate Change Signal (synthetic demo)', fontsize=13)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(annual_pr.index.year, annual_pr.values, color='steelblue', alpha=0.8, width=0.8)
axes[1].axvline(2015, color='gray', ls='--', lw=0.8)
axes[1].set_ylabel('Annual precipitation (mm/year)')
axes[1].set_title('Annual Precipitation')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/CDS_download.png', dpi=100, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/CDS_download.png')

Plot saved to /tmp/CDS_download.png
